# UFC 1 Walkthrough: Systems A/B/C/D/E
Compact notebook: all heavy logic is imported from `elo_calculator.application.ranking.ufc1_walkthrough`.


## 1) Imports + Build Shared Context


In [12]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

ROOT_STR = str(ROOT.resolve())
if ROOT_STR not in sys.path:
    sys.path.insert(0, ROOT_STR)

import elo_calculator.application.ranking.ufc1_walkthrough as ufc1_walkthrough  # noqa: E402, PLR0402

importlib.reload(ufc1_walkthrough)

build_ufc1_context = ufc1_walkthrough.build_ufc1_context
coverage_dataframe = ufc1_walkthrough.coverage_dataframe
order_dataframe = ufc1_walkthrough.order_dataframe
post_ufc1_rankings = ufc1_walkthrough.post_ufc1_rankings
precheckpoint_dataframe = ufc1_walkthrough.precheckpoint_dataframe
record_nc_samples = ufc1_walkthrough.record_nc_samples
run_system_a = ufc1_walkthrough.run_system_a
run_system_b = ufc1_walkthrough.run_system_b
run_system_c = ufc1_walkthrough.run_system_c
run_system_d = ufc1_walkthrough.run_system_d
run_system_e = ufc1_walkthrough.run_system_e
system_a_ps_margin_impact = ufc1_walkthrough.system_a_ps_margin_impact

context = build_ufc1_context(data_dir=ROOT / 'seeder_data' / 'data')
print(f'Event: {context.event_name} ({context.event_date})')

Event: UFC 1: The Beginning (1993-11-12)


## 2) Same-Day Bout Order (record-before driven)


In [2]:
order_df = order_dataframe(context)
print(order_df.to_string(index=False))

 fight_order         fight_id                        matchup tapology_bout_id prefight_a prefight_b  stage_from_record_before
           1 cecdc0da584274b9  Royce Gracie vs Art Jimmerson             1206        0-0        0-0                         0
           2 567a09fd200cfa05   Gerard Gordeau vs Teila Tuli             1190        0-0        0-0                         0
           3 00b0796724ec1c09 Jason DeLucia vs Trent Jenkins             1194        0-0        0-0                         0
           4 2d2bbc86e941e05c   Kevin Rosier vs Zane Frazier             1199        0-0        0-0                         0
           5 46acd54cc0c905fb  Ken Shamrock vs Patrick Smith             1210        3-0        0-0                         0
           6 ac7ca2ec38b96c1a Gerard Gordeau vs Kevin Rosier             1214        1-0        1-0                         1
           7 ffd16691c4c4aafc   Royce Gracie vs Ken Shamrock             1218        1-0        4-0                   

## 3) Record Format Check (NC examples)


In [3]:
nc_examples = record_nc_samples(context, limit=20)
print('Sample records containing NC markers:')
for sample in nc_examples:
    print(' -', sample)

Sample records containing NC markers:
 - 0-1-0 (1 NC)
 - 1-0-0 (1 NC)
 - 1-2-0 (1 NC)
 - 1-5-0 (1 NC)
 - 10-10-0 (1 NC)
 - 10-16-1 (1 NC)
 - 10-3-0 (1 NC)
 - 10-4-0 (1 NC)
 - 10-5-0 (1 NC)
 - 10-6-0 (1 NC)
 - 10-6-1 (1 NC)
 - 11-0-0 (1 NC)
 - 11-10-0 (3 NC)
 - 11-18-0 (1 NC)
 - 11-2-0 (1 NC)
 - 11-22-0 (1 NC)
 - 11-3-0 (1 NC)
 - 11-3-1 (1 NC)
 - 11-5-0 (1 NC)
 - 11-6-0 (1 NC)


## 4) Pre-Checkpoint Coverage For UFC 1 Fighters


In [4]:
pre_df = precheckpoint_dataframe(context)
coverage_df = coverage_dataframe(context)

print('Pre-checkpoint bouts that feed priors:')
print(pre_df.to_string(index=False) if not pre_df.empty else 'No pre-checkpoint bouts found')
print()
print('Coverage summary by fighter:')
print(coverage_df.to_string(index=False))

Pre-checkpoint bouts that feed priors:
event_date      sport        fighter result            opponent record_before                   event_name
1989-01-14     boxing   Kevin Rosier   loss     Andrew Matthews          None         Pedroza vs. Stephens
1989-02-28     boxing   Kevin Rosier   loss      Curtis Jackson          None             Salud vs. Adames
1989-03-02     boxing   Kevin Rosier   loss        John Elacqua          None              Dryer vs. Owens
1989-05-14 kickboxing   Kevin Rosier    win         Don Nielsen          None AJKF: Clash Of The Century 3
1990-01-20 kickboxing   Kevin Rosier   loss       Maurice Smith          None       AJKF: Inspiring Wars 1
1990-04-07     boxing   Kevin Rosier   loss         David Dixon          None            Espinoza vs. Paez
1990-04-29     boxing  Art Jimmerson   loss      Andrew Maynard          None              DeWitt vs. Benn
1990-11-12     boxing   Kevin Rosier    win           Lee Moore          None        Rangel vs. Richardso

## 5) System A (Elo + PS)
MMA pre-checkpoint bouts update priors. UFC 1 bouts then update both fighters in ordered sequence.


In [5]:
a_prior, a_fights, p_a_map, a_states = run_system_a(context)

print('System A: prior updates from pre-checkpoint MMA bouts')
print(a_prior.to_string(index=False) if not a_prior.empty else 'No MMA pre-checkpoint updates')
print()
print('System A: UFC 1 per-fighter before/after rows')
print(a_fights.to_string(index=False))

System A: prior updates from pre-checkpoint MMA bouts
event_date sport      fighter         opponent result  pre_rating  post_rating  delta
1993-09-21   mma Ken Shamrock Masakatsu Funaki    win     1500.00      1505.91 5.9062
1993-10-14   mma Ken Shamrock  Kazuo Takahashi    win     1505.91      1511.73 5.8283
1993-11-08   mma Ken Shamrock      Takaku Fuke    win     1511.73      1517.49 5.7515

System A: UFC 1 per-fighter before/after rows
 fight_order         fight_id        fighter       opponent outcome  pre_rating  post_rating    delta  expected_win_prob  target_score  k_effective  ps_margin
           1 cecdc0da584274b9   Royce Gracie  Art Jimmerson     win     1500.00      1510.35  10.3489             0.5000        0.8098      33.4096     0.0198
           1 cecdc0da584274b9  Art Jimmerson   Royce Gracie    loss     1500.00      1489.65 -10.3489             0.5000        0.1902      33.4096    -0.0198
           2 567a09fd200cfa05 Gerard Gordeau     Teila Tuli     win     1500.0

## 6) System B (Glicko-2 + PS)
Adds uncertainty (RD) and updates both rating and RD for each fighter.


In [6]:
b_prior, b_fights, p_b_map, b_states = run_system_b(context)

print('System B: prior updates from pre-checkpoint MMA bouts')
print(b_prior.to_string(index=False) if not b_prior.empty else 'No MMA pre-checkpoint updates')
print()
print('System B: UFC 1 per-fighter before/after rows (with RD)')
print(b_fights.to_string(index=False))

System B: prior updates from pre-checkpoint MMA bouts
event_date sport      fighter         opponent result  pre_rating  post_rating   delta
1993-09-21   mma Ken Shamrock Masakatsu Funaki    win     1500.00      1590.89 90.8941
1993-10-14   mma Ken Shamrock  Kazuo Takahashi    win     1590.89      1639.14 48.2509
1993-11-08   mma Ken Shamrock      Takaku Fuke    win     1639.14      1669.63 30.4856

System B: UFC 1 per-fighter before/after rows (with RD)
 fight_order         fight_id        fighter       opponent outcome  pre_rating  post_rating     delta  expected_win_prob  target_score  k_effective  ps_margin   pre_rd  post_rd
           1 cecdc0da584274b9   Royce Gracie  Art Jimmerson     win     1500.00      1600.55  100.5538             0.5000        0.8098          0.0     0.0198 350.0000 290.3190
           1 cecdc0da584274b9  Art Jimmerson   Royce Gracie    loss     1500.00      1399.45 -100.5538             0.5000        0.1902          0.0    -0.0198 350.0000 290.3190
       

## 7) System C (Dynamic Factor Bradley-Terry)
Uses cross-sport pre-checkpoint bouts to shift MMA + striking/grappling/durability latents.


In [7]:
c_prior, c_fights, p_c_map, c_states = run_system_c(context)

print('System C: pre-checkpoint latent updates (all sports)')
print(c_prior.to_string(index=False) if not c_prior.empty else 'No pre-checkpoint updates')
print()
print('System C: UFC 1 per-fighter before/after latent rows')
print(c_fights.to_string(index=False))

System C: pre-checkpoint latent updates (all sports)
event_date      sport        fighter            opponent result  pre_mma_rating  post_mma_rating  delta_mma  pre_striking  post_striking  pre_grappling  post_grappling  pre_durability  post_durability
1989-01-14     boxing   Kevin Rosier     Andrew Matthews   loss         1500.00          1500.00     0.0000        0.0000        -0.3704         0.0000          0.0000          0.0000          -0.0617
1989-02-28     boxing   Kevin Rosier      Curtis Jackson   loss         1500.00          1500.00     0.0000       -0.3704        -0.6357         0.0000          0.0000         -0.0617          -0.1059
1989-03-02     boxing   Kevin Rosier        John Elacqua   loss         1500.00          1500.00     0.0000       -0.6357        -0.8316         0.0000          0.0000         -0.1059          -0.1385
1989-05-14 kickboxing   Kevin Rosier         Don Nielsen    win         1500.00          1500.00     0.0000       -0.8316        -0.4322       

## 8) System D (Stacked Logit Mixture)
Combines A/B/C probabilities into a single probability stream.


In [8]:
d_df = run_system_d(context, p_a_map, p_b_map, p_c_map)
print(d_df.to_string(index=False))

 fight_order        fighter       opponent outcome    p_A    p_B    p_C    p_D  actual_minus_pD       mode
           1   Royce Gracie  Art Jimmerson     win 0.5000 0.5000 0.5640 0.5213           0.4787 avg(A,B,C)
           1  Art Jimmerson   Royce Gracie    loss 0.5000 0.5000 0.4360 0.4787          -0.4787 avg(A,B,C)
           2 Gerard Gordeau     Teila Tuli     win 0.5000 0.5000 0.4722 0.4907           0.5093 avg(A,B,C)
           2     Teila Tuli Gerard Gordeau    loss 0.5000 0.5000 0.5278 0.5093          -0.5093 avg(A,B,C)
           3  Jason DeLucia  Trent Jenkins     win 0.5000 0.5000 0.5668 0.5223           0.4777 avg(A,B,C)
           3  Trent Jenkins  Jason DeLucia    loss 0.5000 0.5000 0.4332 0.4777          -0.4777 avg(A,B,C)
           4   Kevin Rosier   Zane Frazier     win 0.5000 0.5000 0.5565 0.5188           0.4812 avg(A,B,C)
           4   Zane Frazier   Kevin Rosier    loss 0.5000 0.5000 0.4435 0.4812          -0.4812 avg(A,B,C)
           5   Ken Shamrock  Patrick 

## 9) System E (Expected Win Rate)
Transforms pairwise probabilities into pool-based expected win rate ranking.


In [9]:
e_movement, e_ranking = run_system_e(context)

print('System E: per-fight EWR movement (both fighters shown)')
print(e_movement.to_string(index=False))
print()
print('System E: UFC 1 pool ranking snapshot')
print(e_ranking.to_string(index=False))

System E: per-fight EWR movement (both fighters shown)
 fight_order        fighter       opponent outcome  ewr_before  ewr_after  ewr_delta
           1   Royce Gracie  Art Jimmerson     win      0.5013     0.5508     0.0495
           1  Art Jimmerson   Royce Gracie    loss      0.4625     0.3809    -0.0816
           2 Gerard Gordeau     Teila Tuli     win      0.3958     0.5021     0.1064
           2     Teila Tuli Gerard Gordeau    loss      0.5054     0.4206    -0.0848
           3  Jason DeLucia  Trent Jenkins     win      0.5026     0.5674     0.0648
           3  Trent Jenkins  Jason DeLucia    loss      0.5026     0.4377    -0.0649
           4   Kevin Rosier   Zane Frazier     win      0.4617     0.5839     0.1221
           4   Zane Frazier   Kevin Rosier    loss      0.5026     0.4362    -0.0664
           5   Ken Shamrock  Patrick Smith     win      0.6648     0.6855     0.0207
           5  Patrick Smith   Ken Shamrock    loss      0.5024     0.4682    -0.0342
          

## 10) Elo PS-Margin / Method Impact Check
Shows UFC 1 deltas with actual PS margin versus a counterfactual where `ps_margin = 0`.


In [10]:
impact_df = system_a_ps_margin_impact(context)

abs_gain_sum = float(impact_df['delta_gain_from_ps'].abs().sum())
if abs_gain_sum == 0.0:
    raise RuntimeError(
        'PS-margin impact is all zeros. Rerun Cell 1 (module reload), then rerun all cells from the top.'
    )

print(f'Total absolute PS impact across fighters: {abs_gain_sum:.4f}')
print(impact_df.to_string(index=False))

Total absolute PS impact across fighters: 57.7520
 fight_order        fighter       opponent outcome method  ps_margin  target_score  target_score_no_ps  delta_with_ps  delta_no_ps  delta_gain_from_ps
           1   Royce Gracie  Art Jimmerson     win    SUB     0.0198        0.8098              0.8054        10.3489      10.1231              0.2258
           1  Art Jimmerson   Royce Gracie    loss    SUB    -0.0198        0.1902              0.1946       -10.3489     -10.1231             -0.2258
           2 Gerard Gordeau     Teila Tuli     win    TKO     0.3023        0.8756              0.8091        13.9566      10.2469              3.7097
           2     Teila Tuli Gerard Gordeau    loss    TKO    -0.3023        0.1244              0.1909       -13.9566     -10.2469             -3.7097
           3  Jason DeLucia  Trent Jenkins     win    SUB     0.2530        0.8639              0.8083        13.2838      10.2181              3.0657
           3  Trent Jenkins  Jason DeLucia  

## 11) Final Ranking Of The 10 Fighters After UFC 1
Side-by-side snapshots after processing the event in fight order.


In [11]:
rankings = post_ufc1_rankings(
    context=context,
    system_a_states=a_states,
    system_b_states=b_states,
    system_c_states=c_states,
    system_e_ranking=e_ranking,
)

print('System A ranking (rating):')
print(rankings['system_a'].to_string(index=False))
print()
print('System B ranking (conservative = rating - RD):')
print(rankings['system_b'].to_string(index=False))
print()
print('System C ranking (conservative = mma_rating - mma_uncertainty):')
print(rankings['system_c'].to_string(index=False))
print()
print('System E ranking (adjusted_score):')
print(rankings['system_e'].to_string(index=False))

System A ranking (rating):
       fighter  rating  mma_fights_count
  Royce Gracie 1537.40                 3
Gerard Gordeau 1520.16                 3
  Ken Shamrock 1515.67                 5
 Jason DeLucia 1513.28                 1
  Kevin Rosier 1494.35                 2
 Art Jimmerson 1489.65                 1
 Patrick Smith 1489.44                 1
 Trent Jenkins 1486.72                 1
    Teila Tuli 1486.04                 1
  Zane Frazier 1484.77                 1

System B ranking (conservative = rating - RD):
       fighter  rating     rd  conservative_rating  mma_fights_count
  Royce Gracie 1805.21 211.64              1593.57                 3
Gerard Gordeau 1665.33 215.58              1449.75                 3
  Ken Shamrock 1626.98 194.70              1432.28                 5
 Jason DeLucia 1618.14 290.32              1327.82                 1
  Kevin Rosier 1501.85 247.47              1254.38                 2
 Patrick Smith 1451.47 279.56              1171.91          